**Bibliometric Analysis of Retracted Publications**

All Python code in this notebook strictly adheres to the [PEP 8 Style Guide](https://peps.python.org/pep-0008/) and is organized into modular sections. Each section is designed to be self-contained, allowing you to run all its cells step-by-step directly in Google Colab.

# **Setup & Configuration**

## **Library Imports**

In [72]:
#pip install fastparquet

***PEP 8 Note.***

1. Imports should be grouped in the following order:

- Standard library imports.
- Related third party imports.
- Local application/library specific imports.

You should put a blank line between each group of imports.

2. Sort imports alphabetically inside each group (e.g., os → pathlib → re) for fast lookup and maintenance.

In [73]:
import numpy as np
import os
from pathlib import Path
import re

import pandas as pd
import requests

## **Constants & Configuration**

***PEP 8 Note.***

Constants are usually defined on a module level and written in all capital letters with underscores separating words. Examples include `MAX_OVERFLOW` and `TOTAL`.

In [74]:
BASE_DIR = Path.cwd().parent

OUTPUT_PATH = BASE_DIR / "models"

DATASET_PATHS = {
    "non_rwd": BASE_DIR / "raw-data" / "non_rwd.csv",
    "rwd": BASE_DIR / "raw-data" / "rwd.csv",
    "reasons": BASE_DIR / "raw-data" / "reason.csv"
}

## **Data Loading**

***Naming Conventions by Stage.***

**Stage 1: Raw Data (Directly from CSV)**

Use `df_raw` to store the dataset immediately after loading it from Google Sheets, keeping the original source data completely untouched.

**Stage 2: Intermediate Preprocessing (Step-by-Step)**

Use action-based variable names such as `df_filtered` or `df_deduped` to represent each specific intermediate transformation step.

**Stage 3: Cleaned/Final Data**

Assign the fully cleaned dataset to `df`, which serves as your production-ready DataFrame for all downstream analysis.

Instead of calling `load_gsheet_to_df()` 3 separate times and writing repetitive assignment statements, you can use **Sequence Unpacking** combined with a **Generator Expression** to fetch and assign all your DataFrames in a single, elegant line of code. This keeps your code short and saves memory while loading.

In [75]:
df_non_rwd_raw, df_rwd_raw, df_reason_raw = (
    pd.read_csv(path)
        for path in DATASET_PATHS.values()
)

# **Data Preprocessing**

## **Data Profiling**

### ***Dataset Overview***

##### **`df_non_rwd_raw`**

In [141]:
df_non_rwd_raw.head()

,authors,author_full_names,authors_id,title,year,source_title,volume,issue,art._no.,page_start,...,coden,pubmed_id,language_of_original_document,abbreviated_source_title,document_type,publication_stage,open_access,source,eid,page_count
0,li w,"li, wei",57223050090,reform and innovation of higher fine arts dist...,2021,journal of physics: conference series,1852,3,32026,NaN,...,NaN,NaN,english,j. phys. conf. ser.,retracted,final,all open access; gold open access,scopus,2-s2.0-85104637686,0
1,chen m; ma h; wang h,"chen, meina; ma, he; wang, hui",57215903533; 57216579080; 57214937749,the coordinated development of technological i...,2021,acm international conference proceeding series,0,NaN,3465679,NaN,...,NaN,NaN,english,acm int. conf. proc. ser.,retracted,final,NaN,scopus,2-s2.0-85113369726,0
2,vivek c; gowtham a; dharaneesh ks; sajina t...,"vivek, c.; gowtham, a.; dharaneesh, k.s.; saji...",55984362100; 60591110600; 58264441700; 5826444...,hyperspectral information in urban areas using...,2023,2023 9th international conference on advanced ...,0,NaN,NaN,1031,...,NaN,NaN,english,"int. conf. adv. comput. commun. syst., icaccs",retracted,final,NaN,scopus,2-s2.0-85159764971,0
3,ali abdu na; wang z,"ali abdu, nail adeeb; wang, zhaoshun",57423073700; 23502254100,retraction:grid based blockchain framework for...,2022,journal of intelligent and fuzzy systems,43,4,NaN,5325,...,NaN,NaN,english,j. intelligent fuzzy syst.,retracted,final,NaN,scopus,2-s2.0-85136783095,0
4,wang w,"wang, wei",57785529500,"recognition, processing, and detection of sens...",2022,journal of sensors,2022,NaN,6900912,NaN,...,NaN,NaN,english,j. sensors,retracted,final,all open access; hybrid gold open access,scopus,2-s2.0-85129035244,0


In [142]:
df_non_rwd_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 584 entries, 0 to 583
Data columns (total 46 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   authors                        582 non-null    object 
 1   author_full_names              582 non-null    object 
 2   authors_id                     577 non-null    object 
 3   title                          584 non-null    object 
 4   year                           584 non-null    int64  
 5   source_title                   584 non-null    object 
 6   volume                         584 non-null    int64  
 7   issue                          160 non-null    object 
 8   art._no.                       452 non-null    object 
 9   page_start                     114 non-null    object 
 10  page_end                       103 non-null    object 
 11  cited_by                       584 non-null    int64  
 12  doi                            583 non-null    obj

***Review.***

- Existing column headers follow standard English grammar with spaces, capitalization, and special characters (e.g., `Author(s) ID`), which require transformation into clean snake_case (e.g., `author_ids`) by removing special symbols and spaces.

- Author-centric fields (`Authors, Author full names, Author(s) ID, and Authors with affiliations`) use semicolons (;) to separate entries, preserving identical element counts per row where `Authors with affiliations` explicitly duplicates shared institution names for each person.

- `Author(s) ID` provides a unique identifier per author, while `DOI` uniquely identifies each paper once web prefixes (such as `https://` or `www.`) are stripped out.

- The text-heavy `References` column can be evaluated for citation volume, but raw reference strings do not explain retraction causes and may face link-rot or access restrictions post-retraction.

- `Publisher` values are stored entirely in lowercase strings, requiring conversion to Capitalized format to ensure readable visualizations in downstream analysis.

- `Language of Original Document` (100% "english") and `Document Type` (100% "retracted" or "retracted publication") remain constant across all 584 records, providing zero analytical variance and making them immediate candidates for deletion.

- Therefore, retraction analysis only requires 8 essential features (`Authors, Author full names, Author(s) ID, Year, Cited by, DOI, Authors with affiliations,` and `Publisher`), allowing all other 38 columns to be safely dropped.

- Among the 8 selected columns, minor missingness exists, which must be resolved by dropping missing key identifiers or imputing textual fields with "Unknown".

##### **`df_rwd_raw`**

In [143]:
df_rwd_raw.head()

,record_id,title,subject,institution,journal,publisher,country,author,urls,articletype,retractiondate,retractiondoi,retractionpubmedid,originalpaperdate,originalpaperdoi,originalpaperpubmedid,retractionnature,reason,paywalled,notes
0,71782,"The study of structural, magnetic and dielectr...",(PHY) Crystallography/Spectroscopy;(PHY) Mater...,"Department of Chemistry, Government Murray Col...",Applied Physics A,Springer - Nature Publishing Group,Egypt;Pakistan;Saudi Arabia;South Korea,Kashif Younas Butt;Salma Aman;Abeer A AlObaid;...,NaN,Research Article;,8/8/2024 0:00,10.1007/s00339-024-07754-7,0.0,8/27/2021 0:00,10.1007/s00339-021-04827-9,0.0,Correction,Error in Image;Updated to Retraction;,No,See also: https://pubpeer.com/publications/C4B...
1,71781,"The study of structural, magnetic and dielectr...",(PHY) Crystallography/Spectroscopy;(PHY) Mater...,"Department of Chemistry, Government Murray Col...",Applied Physics A,Springer - Nature Publishing Group,Egypt;Pakistan;Saudi Arabia;South Korea,Kashif Younas Butt;Salma Aman;Abeer A AlObaid;...,NaN,Research Article;,7/22/2025 0:00,10.1007/s00339-025-08780-9,0.0,8/27/2021 0:00,10.1007/s00339-021-04827-9,0.0,Retraction,Concerns/Issues about Data;Concerns/Issues abo...,No,See also: https://pubpeer.com/publications/C4B...
2,71780,Tunable decorated flake interlayers of functio...,(PHY) Energy;(PHY) Materials Science;(PHY) Nan...,"Department of Physics, College Science, Prince...",Applied Physics A,Springer - Nature Publishing Group,Pakistan;Saudi Arabia;Turkey,Nada Alfryyan;Sumaira Manzoor;Abdul Ghafoor Ab...,NaN,Research Article;,8/5/2025 0:00,10.1007/s00339-025-08843-x,0.0,6/5/2022 0:00,10.1007/s00339-022-05707-6,0.0,Retraction,Concerns/Issues about Data;Concerns/Issues abo...,No,See also: https://pubpeer.com/publications/2DF...
3,71779,Novel Sr-based Al2O4 spinel material an enviro...,(PHY) Engineering - Electrical;(PHY) Materials...,"Institute of Physics, Khwaja Fareed University...",Applied Physics A,Springer - Nature Publishing Group,Pakistan;Saudi Arabia,Salma Aman;Soumaya Gouadria;F F Alharbi;Muhamm...,NaN,Research Article;,8/8/2025 0:00,10.1007/s00339-025-08828-w,0.0,4/15/2023 0:00,10.1007/s00339-023-06591-4,0.0,Retraction,Concerns/Issues about Data;Investigation by Th...,No,NaN
4,71778,The effect of silicon on cerium zirconates pyr...,(PHY) Crystallography/Spectroscopy;(PHY) Engin...,"Institute of Chemical Sciences, Bahauddin Zaka...",Applied Physics A,Springer - Nature Publishing Group,Pakistan;Saudi Arabia,Saleem Mumtaz;Muhammad Naeem Ashiq;Bashir Ahma...,NaN,Research Article;,8/18/2025 0:00,10.1007/s00339-025-08869-1,0.0,11/10/2021 0:00,10.1007/s00339-021-05040-4,0.0,Retraction,Concerns/Issues about Data;Concerns/Issues abo...,No,See also: https://pubpeer.com/publications/8B0...


In [144]:
df_rwd_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71142 entries, 0 to 71141
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   record_id              71142 non-null  int64  
 1   title                  71142 non-null  object 
 2   subject                71142 non-null  object 
 3   institution            71141 non-null  object 
 4   journal                71142 non-null  object 
 5   publisher              71142 non-null  object 
 6   country                71142 non-null  object 
 7   author                 71142 non-null  object 
 8   urls                   39475 non-null  object 
 9   articletype            71142 non-null  object 
 10  retractiondate         71142 non-null  object 
 11  retractiondoi          70652 non-null  object 
 12  retractionpubmedid     65999 non-null  float64
 13  originalpaperdate      71142 non-null  object 
 14  originalpaperdoi       68612 non-null  object 
 15  or

***Review.***

- Column headers use concatenated CamelCase without spaces or underscores (e.g., `OriginalPaperDOI`), requiring conversion to standard snake_case (e.g., `original_paper_doi`).

- `OriginalPaperDOI` shares the exact same format as `DOI` in **`df_non_rwd_raw`**, serving as the primary key to link both datasets together.

- `Author` and `Institution` lack author-to-affiliation alignment, so we drop them here and rely on **`df_non_rwd_raw`**'s structured `Authors with affiliations`.

- Publisher and Journal are cleanly formatted with proper title casing and spacing, so we retain them here and drop **`df_non_rwd_raw`**'s lowercased publisher data.

- `RetractionDate` and `OriginalPaperDate` are stored as strings (m/d/yyyy - h:mm) requiring conversion to datetime, followed by creating a new `retraction_lag_days` column to measure the time gap from publication to retraction.

- `RetractionNature` contains non-retraction events (such as `Correction`); we must filter specifically for `Retraction` records to align with **`df_non_rwd_raw`**.

- The `Reason` column uses internal and trailing semicolons (;) to list multiple causes per record, requiring string stripping and splitting for category analysis.

- Select only essential columns (`OriginalPaperDOI, Journal, Publisher, Country, RetractionDate, OriginalPaperDate, RetractionNature, Reason`) and handle missing values (such as dropping rows missing `OriginalPaperDOI`).

##### **`df_reason_raw`**

To group all reason tags in the Retraction Watch Database (RWD), we first defined main categories following [Tiwari et al. (2026)](https://publicationsdrdo.in/index.php/djlit/article/view/21055) and [Okyay et al. (2025)](https://jkms.org/DOIx.php?id=10.3346/jkms.2025.40.e300). We then mapped the extracted reason tags into a new column, `reason_category`, for analysis.

In [145]:
df_reason_raw.head()

,reason_tag,reason_category
0,Author Unresponsive,Transparency Issues
1,Breach of Policy by Author,Authorship Issues
2,Compromised Peer Review,Peer Review Issues
3,Computer-Aided Content or Computer-Generated C...,Randomly Generated Content
4,Concerns/Issues about Article,Data Concerns


In [146]:
df_reason_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   reason_tag       37 non-null     object
 1   reason_category  37 non-null     object
dtypes: object(2)
memory usage: 724.0+ bytes


In [147]:
df_reason_summary = (
    df_reason_raw.groupby("reason_category")["reason_tag"]
    .apply(lambda tags: "; ".join(tags.dropna().unique()))
    .reset_index(name="reason_tags")
)

df_reason_summary

,reason_category,reason_tags
0,Authorship Issues,Breach of Policy by Author; Concerns/Issues ab...
1,Data Concerns,Concerns/Issues about Article; Concerns/Issues...
2,Duplication,Duplication of/in Article
3,Error,Error by Journal/Publisher; Error in Data
4,Ethical Issues,Concerns/Issues about Human Subject Welfare; I...
5,Fraud,Paper Mill; Rogue Editor
6,Legal Issues,Concerns/Issues about Third Party Involvement;...
7,Others,Date of Article and/or Notice Unknown; Removed...
8,Peer Review Issues,Compromised Peer Review; Concerns/Issues about...
9,Plagiarism,Plagiarism of/in Article; Taken from Dissertat...


## **Data Cleaning & Transformation**

### **Text Standardization**

In this step, we clean and unify the column headers across all incoming datasets as they are loaded into memory. Specifically, we format the column names to enforce a standard `snake_case` layout by:

- Lowercasing all text to eliminate casing inconsistencies across different source files.

- Stripping clutter such as parentheses, special characters, and trailing whitespace.

- Replacing spaces and hyphens with clean single underscores.

In [852]:
def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize DataFrame column names to lowercase snake_case text.

    Args:
        df (pd.DataFrame): The input DataFrame whose header columns
            need whitespace stripped and text converted to snake_case.

    Returns:
        pd.DataFrame: The DataFrame with cleaned, standardized column names.
    """
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace(r"[()]", "", regex=True)
    )
    return df

In [853]:
df_non_rwd_txt_std, df_rwd_txt_std, df_reason_txt_std = (
    standardize_column_names(df)
    for df in (df_non_rwd_raw, df_rwd_raw, df_reason_raw)
)

In [854]:
for df in (df_non_rwd_txt_std, df_rwd_txt_std, df_reason_txt_std):
    print(df.columns.tolist())
    print("-" * 50)

['authors', 'author_full_names', 'authors_id', 'title', 'year', 'source_title', 'volume', 'issue', 'art._no.', 'page_start', 'page_end', 'cited_by', 'doi', 'link', 'affiliations', 'authors_with_affiliations', 'abstract', 'author_keywords', 'unnamed:_18', 'molecular_sequence_numbers', 'chemicals/cas', 'tradenames', 'manufacturers', 'funding_details', 'funding_texts', 'references', 'correspondence_address', 'editors', 'publisher', 'sponsors', 'conference_name', 'conference_date', 'conference_location', 'conference_code', 'issn', 'isbn', 'coden', 'pubmed_id', 'language_of_original_document', 'abbreviated_source_title', 'document_type', 'publication_stage', 'open_access', 'source', 'eid', 'page_count']
--------------------------------------------------
['record_id', 'title', 'subject', 'institution', 'journal', 'publisher', 'country', 'author', 'urls', 'articletype', 'retractiondate', 'retractiondoi', 'retractionpubmedid', 'originalpaperdate', 'originalpaperdoi', 'originalpaperpubmedid', '

The **`df_rwd_txt_std`** dataset used concatenated words without spaces or separators (such as `originalpaperdoi`). To fix this, we applied an explicit column renaming mapping to insert necessary underscores and standardize key identifiers (e.g., converting `originalpaperdoi` to `original_paper_doi`), ensuring uniform syntax across all pipeline datasets.

In [855]:
rwd_column_renames = {
    "retractiondate": "retraction_date",
    "retractiondoi": "retraction_doi",
    "originalpaperdate": "publication_date",
    "originalpaperdoi": "original_paper_doi",
    "retractionnature": "retraction_nature",
}

df_rwd_txt_std = df_rwd_txt_std.rename(columns=rwd_column_renames)

In [856]:
print("=== Updated df_rwd_txt_std Columns ===")
print(df_rwd_txt_std.columns.tolist())

=== Updated df_rwd_txt_std Columns ===
['record_id', 'title', 'subject', 'institution', 'journal', 'publisher', 'country', 'author', 'urls', 'articletype', 'retraction_date', 'retraction_doi', 'retractionpubmedid', 'publication_date', 'original_paper_doi', 'originalpaperpubmedid', 'retraction_nature', 'reason', 'paywalled', 'notes']


### **Column Extraction**

In the **Column Extraction** step, we isolated core target features such as article identifiers, author affiliations, publication dates, and retraction reasons from the larger standardized datasets. By slicing these specific fields into focused DataFrames, we eliminated unneeded variables and reduced memory overhead, creating clean, lightweight subsets optimized for scientometric analysis.

In [857]:
non_rwd_col_extracted = [
    "author_full_names",
    "authors_id",
    "doi",
    "authors_with_affiliations"
]

rwd_col_extracted = [
    "journal",
    "publisher",
    "subject",
    "retraction_date",
    "publication_date",
    "original_paper_doi",
    "retraction_nature",
    "reason",
]

df_non_rwd_col_extracted = df_non_rwd_txt_std[non_rwd_col_extracted].copy()
df_rwd_col_extracted = df_rwd_txt_std[rwd_col_extracted].copy()
df_reason_col_extracted = df_reason_txt_std.copy()

In [858]:
print("=== Extraction Summary ===")
for df in (df_non_rwd_col_extracted, df_rwd_col_extracted, df_reason_col_extracted):
    print()
    print(df.columns.tolist())
    print("-" * 50)

=== Extraction Summary ===

['author_full_names', 'authors_id', 'doi', 'authors_with_affiliations']
--------------------------------------------------

['journal', 'publisher', 'subject', 'retraction_date', 'publication_date', 'original_paper_doi', 'retraction_nature', 'reason']
--------------------------------------------------

['reason_tag', 'reason_category']
--------------------------------------------------


### **Type Conversion**

In [859]:
df_non_rwd_col_extracted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 584 entries, 0 to 583
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   author_full_names          582 non-null    object
 1   authors_id                 577 non-null    object
 2   doi                        583 non-null    object
 3   authors_with_affiliations  582 non-null    object
dtypes: object(4)
memory usage: 18.4+ KB


In [860]:
df_non_rwd_casted = df_non_rwd_col_extracted.copy()

In [861]:
df_non_rwd_casted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 584 entries, 0 to 583
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   author_full_names          582 non-null    object
 1   authors_id                 577 non-null    object
 2   doi                        583 non-null    object
 3   authors_with_affiliations  582 non-null    object
dtypes: object(4)
memory usage: 18.4+ KB


In [862]:
df_rwd_casted = df_rwd_col_extracted.copy()

date_columns = ["retraction_date", "publication_date"]

for col in date_columns:
    if col in df_rwd_casted.columns:
        df_rwd_casted[col] = pd.to_datetime(
            df_rwd_casted[col], errors="coerce"
        )

df_rwd_casted["original_paper_doi"] = df_rwd_casted["original_paper_doi"].str.lower()

In [863]:
df_rwd_casted.head(10)

,journal,publisher,subject,retraction_date,publication_date,original_paper_doi,retraction_nature,reason
0,Applied Physics A,Springer - Nature Publishing Group,(PHY) Crystallography/Spectroscopy;(PHY) Mater...,2024-08-08,2021-08-27,10.1007/s00339-021-04827-9,Correction,Error in Image;Updated to Retraction;
1,Applied Physics A,Springer - Nature Publishing Group,(PHY) Crystallography/Spectroscopy;(PHY) Mater...,2025-07-22,2021-08-27,10.1007/s00339-021-04827-9,Retraction,Concerns/Issues about Data;Concerns/Issues abo...
2,Applied Physics A,Springer - Nature Publishing Group,(PHY) Energy;(PHY) Materials Science;(PHY) Nan...,2025-08-05,2022-06-05,10.1007/s00339-022-05707-6,Retraction,Concerns/Issues about Data;Concerns/Issues abo...
3,Applied Physics A,Springer - Nature Publishing Group,(PHY) Engineering - Electrical;(PHY) Materials...,2025-08-08,2023-04-15,10.1007/s00339-023-06591-4,Retraction,Concerns/Issues about Data;Investigation by Th...
4,Applied Physics A,Springer - Nature Publishing Group,(PHY) Crystallography/Spectroscopy;(PHY) Engin...,2025-08-18,2021-11-10,10.1007/s00339-021-05040-4,Retraction,Concerns/Issues about Data;Concerns/Issues abo...
5,Behavioral Ecology and Sociobiology,Springer - Nature Publishing Group,(BLS) Zoology;,2025-09-12,2015-06-10,10.1007/s00265-015-1953-6,Retraction,Error in Methods;Original Data and/or Images n...
6,Environmental Research,Elsevier,(PHY) Chemistry;(PHY) Energy;(PHY) Engineering...,2026-07-15,2021-12-01,10.1016/j.envres.2021.112474,Retraction,Compromised Peer Review;Concerns/Issues about ...
7,Multimedia Tools and Applications,Springer - Nature Publishing Group,(B/T) Technology;(HSC) Medicine - Dentistry;(H...,2023-06-07,2023-05-22,10.1007/s11042-023-15581-w,Correction,Error in Text;Updated to Retraction;
8,Multimedia Tools and Applications,Springer - Nature Publishing Group,(B/T) Technology;(HSC) Medicine - Dentistry;(H...,2024-02-27,2023-05-22,10.1007/s11042-023-15581-w,Retraction,Euphemisms for Plagiarism;Investigation by Jou...
9,The American Journal of Clinical Nutrition,Elsevier,(HSC) Medicine - Obstetrics/Gynecology;(HSC) N...,2024-11-19,2024-11-19,10.1016/j.ajcnut.2024.11.014,Retraction,Date of Article and/or Notice Unknown;Error in...


In [864]:
df_rwd_casted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71142 entries, 0 to 71141
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   journal             71142 non-null  object        
 1   publisher           71142 non-null  object        
 2   subject             71142 non-null  object        
 3   retraction_date     71141 non-null  datetime64[ns]
 4   publication_date    71141 non-null  datetime64[ns]
 5   original_paper_doi  68612 non-null  object        
 6   retraction_nature   71142 non-null  object        
 7   reason              71142 non-null  object        
dtypes: datetime64[ns](2), object(6)
memory usage: 4.3+ MB


In [865]:
df_reason_col_extracted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   reason_tag       37 non-null     object
 1   reason_category  37 non-null     object
dtypes: object(2)
memory usage: 724.0+ bytes


In [866]:
df_reason_casted = df_reason_col_extracted.copy()

### **Deduplication**

We checked for duplicates in two steps to make sure no paper was counted twice. We removed exact duplicate rows where every detail was identical, keeping only one copy.

In [867]:
def check_duplicate_record(df: pd.DataFrame) -> None:
    """Checks if a DataFrame contains any exact full-row duplicate records.

    Args:
        df (pd.DataFrame): The input DataFrame to evaluate.
    """
    duplicate_exists = df.duplicated().any()

    if duplicate_exists:
        dup_count = df.duplicated().sum()
        print(f"Warning: Found {dup_count:,} duplicate rows.")
    else:
        print("No duplicate rows detected.")

In [868]:
check_duplicate_record(df_non_rwd_casted)

No duplicate rows detected.


In [869]:
df_non_rwd_deduplicated = df_non_rwd_casted.copy()

In [870]:
check_duplicate_record(df_rwd_casted)

In [871]:
def remove_duplicated_record(df: pd.DataFrame) -> pd.DataFrame:
    """Removes full-row duplicate records from a DataFrame and prints removal metrics.

    Args:
        df (pd.DataFrame): The input DataFrame to clean.

    Returns:
        pd.DataFrame: A clean DataFrame with duplicate records removed and index reset.
    """
    count_before = len(df)

    if not df.duplicated().any():
        print("No duplicate records to remove.")
        return df

    df_cleaned = df.drop_duplicates().reset_index(drop=True)

    count_after = len(df_cleaned)
    count_removed = count_before - count_after

    print(f"\n{'=' * 50}")
    print("Duplicate Record Removal Summary")
    print(f"{'=' * 50}")
    print(f"Records before drop: {count_before:,}")
    print(f"Records removed:     {count_removed:,}")
    print(f"Records remaining:   {count_after:,}")

    return df_cleaned

In [872]:
df_rwd_deduplicated = remove_duplicated_record(df_rwd_casted)


Duplicate Record Removal Summary
Records before drop: 71,142
Records removed:     1,307
Records remaining:   69,835


In [873]:
df_reason_deduplicated = df_reason_casted.copy()

### **Remove non-retraction nature**

In [874]:
len_before = len(df_rwd_deduplicated)

print("--- DỮ LIỆU TRƯỚC KHI LỌC ---")
print(f"Tổng số dòng ban đầu: {len_before}")

df_rwd_deduplicated = df_rwd_deduplicated.query('retraction_nature == "Retraction"').reset_index(drop=True)

len_after = len(df_rwd_deduplicated)
num_removed = len_before - len_after

print("\n--- DỮ LIỆU SAU KHI LỌC ---")
print(f"Tổng số dòng sau khi lọc: {len_after}")
print(f"Số record bị remove: {num_removed}")

--- DỮ LIỆU TRƯỚC KHI LỌC ---
Tổng số dòng ban đầu: 69835

--- DỮ LIỆU SAU KHI LỌC ---
Tổng số dòng sau khi lọc: 64791
Số record bị remove: 5044


### **Handling Missing**

In [875]:
def summarize_missing_values(
    df: pd.DataFrame, dataset_name: str = "DataFrame"
) -> None:
    """Prints missing value counts and percentages for a single DataFrame.

    Treats NaNs, None, exact empty strings (""), and whitespace-only strings as
    missing values.

    Args:
        df (pd.DataFrame): The target DataFrame to inspect for missing values.
        dataset_name (str, optional): Label or variable name of the dataset
            for display headers. Defaults to "DataFrame".
    """
    total_rows = len(df)

    # Temporarily replace empty and whitespace-only strings with np.nan for counting
    df_temp = df.replace(r"^\s*$", np.nan, regex=True)

    missing_count = df_temp.isna().sum()
    missing_pct = (
        (missing_count / total_rows * 100) if total_rows > 0 else 0.0
    )

    summary_df = pd.DataFrame(
        {
            "Missing Count": missing_count,
            "Missing (%)": missing_pct.round(2),
        }
    )

    print(f"\n{'=' * 50}")
    print(f"Dataset: {dataset_name} | Total Rows: {total_rows:,}")
    print(f"{'=' * 50}")

    if summary_df["Missing Count"].sum() == 0:
        print("No missing values found!")
    else:
        print(summary_df)

In [876]:
summarize_missing_values(df_non_rwd_deduplicated)


Dataset: DataFrame | Total Rows: 584
                           Missing Count  Missing (%)
author_full_names                      2         0.34
authors_id                             7         1.20
doi                                    1         0.17
authors_with_affiliations              2         0.34


In [877]:
summarize_missing_values(df_rwd_deduplicated)


Dataset: DataFrame | Total Rows: 64,791
                    Missing Count  Missing (%)
journal                         0         0.00
publisher                       0         0.00
subject                         0         0.00
retraction_date                 1         0.00
publication_date                1         0.00
original_paper_doi           1666         2.57
retraction_nature               0         0.00
reason                          0         0.00


In [878]:
def drop_null_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Drops rows containing any null values or empty strings and logs record counts.

    Treats exact empty strings ("") and whitespace-only strings as NaN
    prior to dropping null rows.

    Args:
        df (pd.DataFrame): The input DataFrame to clean.

    Returns:
        pd.DataFrame: A clean DataFrame with null/empty rows removed and index reset.
    """
    count_before = len(df)

    df_cleaned = df.dropna(how="any").reset_index(drop=True).copy()

    count_after = len(df_cleaned)
    count_dropped = count_before - count_after
    pct_dropped = (
        (count_dropped / count_before * 100) if count_before > 0 else 0.0
    )

    print(f"\n{'=' * 50}")
    print("Record Drop Summary")
    print(f"{'=' * 50}")
    print(f"Records before drop: {count_before:,}")
    print(f"Records after drop:  {count_after:,}")
    print(f"Records dropped:     {count_dropped:,} ({pct_dropped:.2f}%)")

    return df_cleaned

In [879]:
df_non_rwd_drop_null = drop_null_rows(df_non_rwd_deduplicated)


Record Drop Summary
Records before drop: 584
Records after drop:  577
Records dropped:     7 (1.20%)


In [880]:
df_rwd_drop_null = drop_null_rows(df_rwd_deduplicated)


Record Drop Summary
Records before drop: 64,791
Records after drop:  63,124
Records dropped:     1,667 (2.57%)


In [881]:
def remove_unavailable_dois(df: pd.DataFrame, col_name: str = "original_paper_doi"
) -> pd.DataFrame:
    """Removes records where the DOI column is marked as 'unavailable'.

    Normalizes strings (lowercase, stripped) before checking to capture
    variations like 'Unavailable' or ' unavailable '.

    Args:
        df (pd.DataFrame): Input DataFrame to clean.
        col_name (str): Column to check. Defaults to "original_paper_doi".

    Returns:
        pd.DataFrame: A new DataFrame with 'unavailable' records removed.
    """
    records_before = len(df)

    is_unavailable = (
        df[col_name].astype(str).str.strip().str.lower() == "unavailable"
    )

    df_cleaned = df.loc[~is_unavailable].reset_index(drop=True)

    records_after = len(df_cleaned)
    records_dropped = records_before - records_after
    pct_dropped = (
        (records_dropped / records_before * 100) if records_before > 0 else 0.0
    )

    print(f"\n{'=' * 50}")
    print("Record Drop Summary")
    print(f"{'=' * 50}")
    print(f"Records before drop: {records_before:,}")
    print(f"Records after drop:  {records_after:,}")
    print(f"Records dropped:     {records_dropped:,} ({pct_dropped:.2f}%)")

    return df_cleaned

In [882]:
df_rwd_drop_null = remove_unavailable_dois(
    df=df_rwd_drop_null,
    col_name="original_paper_doi"
)


Record Drop Summary
Records before drop: 63,124
Records after drop:  59,934
Records dropped:     3,190 (5.05%)


In [883]:
df_reason_drop_null = df_reason_deduplicated.copy()

### **Remove duplicate retraction events**

In [884]:
df_non_rwd_event_deduplicated = df_non_rwd_drop_null.copy()

We checked for duplicate paper IDs in **`df_rwd_drop_null`** (DOIs) to identify publications listed across multiple records.

In [885]:
def check_column_duplicates(df: pd.DataFrame, col_name: str) -> bool:
    """Checks if a specific column in a DataFrame contains non-unique values.

    Args:
        df (pd.DataFrame): The input DataFrame to inspect.
        col_name (str): The column name to check for duplicate values.

    Returns:
        bool: True if non-unique (duplicate) values exist, False otherwise.

    Raises:
        KeyError: If `col_name` does not exist in `df.columns`.
    """
    if col_name not in df.columns:
        raise KeyError(f"Column '{col_name}' not found in DataFrame.")

    total_rows = len(df)
    total_non_null = df[col_name].count()
    unique_count = df[col_name].nunique(dropna=True)

    duplicate_mask = df[col_name].duplicated(keep=False)
    duplicate_row_count = total_rows - unique_count
    has_duplicates = duplicate_row_count > 0

    pct_duplicated = (
        (duplicate_row_count / total_rows * 100) if total_rows > 0 else 0.0
    )

    print(f"\n{'=' * 50}")
    print(f"Uniqueness Check: Column '{col_name}'")
    print(f"{'=' * 50}")
    print(f"Total Rows:             {total_rows:,}")
    print(f"Non-Null Values:        {total_non_null:,}")
    print(f"Unique Values:          {unique_count:,}")
    print(f"Duplicate Rows Count:   {duplicate_row_count:,} ({pct_duplicated:.2f}%)")
    print(f"Status:                 {'NON-UNIQUE (Duplicates Found)' if has_duplicates else 'UNIQUE (All Values Distinct)'}")

    return has_duplicates

In [886]:
has_rwd_dups = check_column_duplicates(
    df=df_rwd_drop_null,
    col_name="original_paper_doi"
)


Uniqueness Check: Column 'original_paper_doi'
Total Rows:             59,934
Non-Null Values:        59,934
Unique Values:          59,856
Duplicate Rows Count:   78 (0.13%)
Status:                 NON-UNIQUE (Duplicates Found)


In [887]:
duplicate_mask = df_rwd_drop_null.duplicated(
    subset=["original_paper_doi"],
    keep=False
)

df_rwd_duplicates = (
    df_rwd_drop_null[duplicate_mask]
    .sort_values(by="original_paper_doi")
    .copy()
)

df_rwd_duplicates

,journal,publisher,subject,retraction_date,publication_date,original_paper_doi,retraction_nature,reason
49016,JAMA Pediatrics,JAMA Network,(B/T) Business - Marketing;(BLS) Nutrition;(SO...,2017-10-20,2012-10-01,10.1001/archpediatrics.2012.999,Retraction,Breach of Policy by Author;Error in Data;Error...
49969,JAMA Pediatrics,JAMA Network,(B/T) Business - Marketing;(BLS) Nutrition;(SO...,2017-09-21,2012-10-01,10.1001/archpediatrics.2012.999,Retraction,Error in Analyses;Error in Data;Error in Metho...
7391,Economic Change and Restructuring,Springer - Nature Publishing Group,(B/T) Business - Economics;(B/T) Technology;(E...,2024-10-29,2024-05-15,10.1007/s10644-024-09687-w,Retraction,Concerns/Issues about Referencing/Attributions...
8698,Economic Change and Restructuring,Springer - Nature Publishing Group,(B/T) Business - Economics;(B/T) Government;(E...,2024-10-29,2024-05-15,10.1007/s10644-024-09687-w,Retraction,Compromised Peer Review;Investigation by Journ...
53470,Tumor Biology (Tumour Biology) - Official Jour...,Springer,(BLS) Biology - Cancer;(BLS) Biology - Molecul...,2017-04-20,2015-01-06,10.1007/s13277-014-2995-5,Retraction,Compromised Peer Review;Investigation by Journ...
...,...,...,...,...,...,...,...,...
36419,Journal of Investigative Medicine: The Officia...,BMJ Publishing,(HSC) Medicine - Infectious Disease;(HSC) Medi...,2021-04-01,2021-01-25,10.1136/jim-2021-srmc,Retraction,Duplication of/in Article;Removed;
1822,PLoS Genetics,PLoS,(BLS) Biology - Molecular;(BLS) Genetics;,2025-10-27,2007-07-06,10.1371/journal.pgen.0030110,Retraction,Duplication of/in Image;Error in Image;Investi...
1821,PLoS Genetics,PLoS,(BLS) Biology - Molecular;(BLS) Genetics;,2017-02-10,2007-07-06,10.1371/journal.pgen.0030110,Retraction,Duplication of/in Image;Euphemisms for Duplica...
26815,Handbuch für Wirtschaftsarchive,De Gruyter,(B/T) Business - General;,2005-01-01,2005-01-01,10.1524/9783486834062.275,Retraction,Copyright Claims;Date of Article and/or Notice...


As highlighted by [Crossref's integration of Retraction Watch data](https://www.crossref.org/blog/retraction-watch-retractions-now-in-the-crossref-api/?utm_source=chatgpt.com), a single publication can sometimes have more than one retraction record or update. When two entries shared the same paper ID (DOI), we evaluated their retraction dates to combine them into one complete record: if the dates were the same, we kept the date and merged their retraction reasons; if the dates were different, we selected the earliest date to accurately reflect when the paper was first retracted while still merging all listed reasons.

In [888]:
def _merge_reasons(series: pd.Series) -> str:
    """Collects and joins non-empty, unique reasons with '; '.

    Args:
        series (pd.Series): Series containing retraction reason strings for
            a given DOI group.

    Returns:
        str: Semicolon-separated string of unique, non-empty retraction reasons.
    """
    valid_reasons = [
        str(val).strip()
        for val in series.dropna()
        if str(val).strip() != "" and str(val).strip().lower() != "nan"
    ]

    unique_reasons = list(dict.fromkeys(valid_reasons))
    return "; ".join(unique_reasons)

In [889]:
df_rwd_event_deduplicated = df_rwd_drop_null.copy()

In [890]:
agg_rules = {}
for col in df_rwd_event_deduplicated.columns:
    if col == "retraction_date":
        agg_rules[col] = "min"
    elif col in ("reason"):
        agg_rules[col] = _merge_reasons
    else:
        agg_rules[col] = "first"

df_rwd_event_deduplicated = (
    df_rwd_event_deduplicated.groupby("original_paper_doi", as_index=False)
    .agg(agg_rules)
    .reset_index(drop=True)
)

In [891]:
has_rwd_dups = check_column_duplicates(
    df=df_rwd_event_deduplicated,
    col_name="original_paper_doi"
)


Uniqueness Check: Column 'original_paper_doi'
Total Rows:             59,856
Non-Null Values:        59,856
Unique Values:          59,856
Duplicate Rows Count:   0 (0.00%)
Status:                 UNIQUE (All Values Distinct)


In [892]:
df_reason_event_deduplicated = df_reason_drop_null.copy()

### **Data Integration**

In [ ]:
df_rwd_merged = pd.merge(
    df_non_rwd_event_deduplicated,
    df_rwd_event_deduplicated,
    left_on="doi",
    right_on="original_paper_doi",
    how="inner",
    suffixes=("_non_rwd", "_rwd"),
)

--- THỐNG KÊ SAU KHI MERGE (INNER JOIN) ---
Số record ban đầu của non_rwd : 577
Số record ban đầu của rwd     : 59856
Số record sau khi merge (giữ lại): 501
Số record bị remove từ non_rwd: 76
Số record bị remove từ rwd    : 59355



In [894]:
df_rwd_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501 entries, 0 to 500
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   author_full_names          501 non-null    object        
 1   authors_id                 501 non-null    object        
 2   doi                        501 non-null    object        
 3   authors_with_affiliations  501 non-null    object        
 4   journal                    501 non-null    object        
 5   publisher                  501 non-null    object        
 6   subject                    501 non-null    object        
 7   retraction_date            501 non-null    datetime64[ns]
 8   publication_date           501 non-null    datetime64[ns]
 9   original_paper_doi         501 non-null    object        
 10  retraction_nature          501 non-null    object        
 11  reason                     501 non-null    object        
dtypes: datet

In [895]:
target_years = [2021, 2022, 2023, 2024, 2025, 2026]

# Lọc các dòng có năm KHÔNG nằm trong danh sách target_years
df_filtered = df_rwd_merged[
    ~pd.to_datetime(df_rwd_merged["publication_date"])
    .dt.year.isin(target_years)
]

df_filtered

,author_full_names,authors_id,doi,authors_with_affiliations,journal,publisher,subject,retraction_date,publication_date,original_paper_doi,retraction_nature,reason
17,"chen, caixia; geng, liwei; zhou, sheng",56723063600; 26659001900; 57215023388,10.1007/s00521-020-04959-8,"chen c., school of information science and tec...",Neural Computing and Applications,Springer,(B/T) Business - Accounting;(B/T) Computer Sci...,2022-12-29,2020-04-30,10.1007/s00521-020-04959-8,Retraction,Concerns/Issues about Referencing/Attributions...
31,"amir latif, rana m.; hussain, khalid; jhanjhi,...",57208427047; 57661242500; 36088700700; 5520144...,10.1007/s11042-020-10087-1,"amir latif r.m., department of computer scienc...",Multimedia Tools and Applications,Springer - Nature Publishing Group,(B/T) Business - Management;(B/T) Technology;(...,2022-09-14,2020-11-10,10.1007/s11042-020-10087-1,Retraction,Concerns/Issues about Third Party Involvement;...
78,"pon senniah, j.; ram prasad, a.v.",57216613770; 57216617205,10.1007/s12652-020-01862-x,"pon senniah j., sbm college of engineering and...",Journal of Ambient Intelligence and Humanized ...,Springer,(B/T) Computer Science;,2022-07-04,2020-04-21,10.1007/s12652-020-01862-x,Retraction,Compromised Peer Review;Investigation by Journ...
128,"wang, liangang; zhang, feng; du, zhenhong; che...",59649103900; 56434720200; 25929119800; 5965370...,10.1016/j.micpro.2020.103526,"wang l., institute of geographic information s...",Microprocessors and Microsystems,Elsevier,(B/T) Computer Science;(PHY) Geology;,2024-10-20,2020-11-30,10.1016/j.micpro.2020.103526,Retraction,Compromised Peer Review;Investigation by Journ...
153,"li, daming; deng, lianbing; su, qinglang",57195604612; 57195607854; 57214099131,10.1007/s11063-020-10258-z,"li d., the post-doctoral research center of zh...",Neural Processing Letters,Springer,(B/T) Computer Science;(B/T) Technology;,2022-10-26,2020-04-30,10.1007/s11063-020-10258-z,Retraction,Compromised Peer Review;Investigation by Journ...
163,"zhao, shubo; su, zheqian; miao, guoxin",57218123546; 57211181875; 55513000700,10.1177/0020720920940614,"zhao s., college of foreign language teaching ...",The International Journal of Electrical Engine...,SAGE Publications,(B/T) Technology;(SOC) Education;,2023-11-01,2020-07-16,10.1177/0020720920940614,Retraction,Compromised Peer Review;Investigation by Journ...
178,"panneerselvam, r.; singaravel, g.",57215822503; 24449654800,10.1007/s12652-020-01868-5,"panneerselvam r., department of electronics an...",Journal of Ambient Intelligence and Humanized ...,Springer - Nature Publishing Group,(B/T) Technology;(SOC) Communications;,2022-05-30,2020-03-16,10.1007/s12652-020-01868-5,Retraction,Compromised Peer Review;Investigation by Journ...
197,"nivedita, v.; nandhagopal, n.",57208263702; 56568221400,10.1007/s12652-020-01787-5,"nivedita v., star lion college of engineering ...",Journal of Ambient Intelligence and Humanized ...,Springer - Nature Publishing Group,(B/T) Computer Science;(B/T) Technology;(SOC) ...,2022-06-20,2020-03-02,10.1007/s12652-020-01787-5,Retraction,Compromised Peer Review;Investigation by Journ...
211,"eken, suleyman",55516480700,10.1007/s00500-020-05387-5,"eken s., department of information systems eng...",Soft Computing,Springer,(B/T) Computer Science;(B/T) Data Science;(HSC...,2023-05-29,2020-10-19,10.1007/s00500-020-05387-5,Retraction,Concerns/Issues about Referencing/Attributions...
237,"sivaram, m.; kaliappan, m.; shobana, s. jeya; ...",55220262500; 55315684100; 57364963300; 6010648...,10.1007/s12652-020-02082-z,"sivaram m., department of computer networking,...",Journal of Ambient Intelligence and Humanized ...,Springer - Nature Publishing Group,(B/T) Computer Science;,2022-06-07,2020-05-14,10.1007/s12652-020-02082-z,Retraction,Compromised Peer Review;Investigation by Journ...


In [896]:
df_non_rwd_only = df_non_rwd_event_deduplicated[
    ~df_non_rwd_event_deduplicated["doi"].isin(
        df_rwd_event_deduplicated["original_paper_doi"]
    )
].copy()

df_non_rwd_only.info()

<class 'pandas.core.frame.DataFrame'>
Index: 76 entries, 2 to 507
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   author_full_names          76 non-null     object
 1   authors_id                 76 non-null     object
 2   doi                        76 non-null     object
 3   authors_with_affiliations  76 non-null     object
dtypes: object(4)
memory usage: 3.0+ KB


In [907]:
df_rwd_only = df_rwd_event_deduplicated[
    ~df_rwd_event_deduplicated["original_paper_doi"].isin(
        df_non_rwd_event_deduplicated["doi"]
    )
].copy()

len_after = len(df_rwd_only)
num_removed = len_before - len_after

df_rwd_only.info()

<class 'pandas.core.frame.DataFrame'>
Index: 59355 entries, 0 to 59855
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   journal             59355 non-null  object        
 1   publisher           59355 non-null  object        
 2   subject             59355 non-null  object        
 3   retraction_date     59355 non-null  datetime64[ns]
 4   publication_date    59355 non-null  datetime64[ns]
 5   original_paper_doi  59355 non-null  object        
 6   retraction_nature   59355 non-null  object        
 7   reason              59355 non-null  object        
dtypes: datetime64[ns](2), object(6)
memory usage: 4.1+ MB


### **Data Modeling**

In [898]:
retraction_cols = [
    "doi",
    "cited_by",
    "publication_date",
    "retraction_date",
    "journal",
    "publisher",
]

cols_present = [c for c in retraction_cols if c in df_rwd_merged.columns]
retraction = df_rwd_merged[cols_present].copy()

retraction["retraction_lag_days"] = (
        retraction["retraction_date"] - retraction["publication_date"]
    ).dt.days

retraction = retraction.drop_duplicates(subset=["doi"]).reset_index(
    drop=True
)

retraction.head()

,doi,publication_date,retraction_date,journal,publisher,retraction_lag_days
0,10.1088/1742-6596/1852/3/032026,2021-04-13,2022-09-09,Journal of Physics: Conference Series,IOP Publishing,514
1,10.1145/3465631.3465679,2021-08-19,2022-02-24,ICIMTECH 21: The Sixth International Conferenc...,Association for Computing Machinery (ACM),189
2,10.3233/jifs-213414,2022-03-01,2024-08-23,Journal of Intelligent & Fuzzy Systems,IOS Press (bought by Sage November 2023),906
3,10.1155/2022/6900912,2022-04-13,2023-08-23,Journal of Sensors,Hindawi,497
4,10.1155/2022/1125084,2022-01-13,2023-12-29,Security and Communication Networks,Hindawi,715


In [899]:
reason = df_reason_event_deduplicated.copy()

reason.head()

,reason_tag,reason_category
0,Author Unresponsive,Transparency Issues
1,Breach of Policy by Author,Authorship Issues
2,Compromised Peer Review,Peer Review Issues
3,Computer-Aided Content or Computer-Generated C...,Randomly Generated Content
4,Concerns/Issues about Article,Data Concerns


In [900]:
df_temp = df_rwd_merged[["doi", "reason"]].dropna(subset=["reason"]).copy()
df_temp["reason_tag"] = df_temp["reason"].str.split(";")
df_temp = df_temp.explode("reason_tag")
df_temp["reason_tag"] = df_temp["reason_tag"].str.strip()
df_temp = df_temp[df_temp["reason_tag"] != ""]

retraction_reason = df_temp[["doi", "reason_tag"]]

retraction_reason.head()

,doi,reason_tag
0,10.1088/1742-6596/1852/3/032026,Compromised Peer Review
0,10.1088/1742-6596/1852/3/032026,Investigation by Journal/Publisher
0,10.1088/1742-6596/1852/3/032026,Paper Mill
1,10.1145/3465631.3465679,Compromised Peer Review
1,10.1145/3465631.3465679,Investigation by Journal/Publisher


In [901]:
data = df_rwd_merged[["author_full_names", "authors_id"]].dropna()

records = [
    {
        "author_id": author_id.strip(),
        "author_name": author_name.strip().title(),
    }
    for author_names, author_ids in zip(
        data["author_full_names"],
        data["authors_id"],
    )
    for author_name, author_id in zip(
        str(author_names).split(";"),
        str(author_ids).split(";"),
    )
    if author_name.strip() and author_id.strip()
]

author = (
    pd.DataFrame(records)
    .drop_duplicates(subset=["author_id"])
    .reset_index(drop=True)
)

author.head()

,author_id,author_name
0,57223050090,"Li, Wei"
1,57215903533,"Chen, Meina"
2,57216579080,"Ma, He"
3,57214937749,"Wang, Hui"
4,57423073700,"Ali Abdu, Nail Adeeb"


In [902]:
data = df_rwd_merged[
    ["doi", "authors_id", "authors_with_affiliations"]
].dropna()

records = [
    {
        "doi": doi,
        "author_id": author_id.strip(),
        "is_1st_author": author_index == 0,
        "affiliation_country": (
            affiliation.split(",")[-1].strip().title()
            if affiliation
            else None
        ),
    }
    for doi, author_ids, affiliations in zip(
        data["doi"],
        data["authors_id"],
        data["authors_with_affiliations"],
    )
    for author_index, (author_id, affiliation) in enumerate(
        zip(
            str(author_ids).split(";"),
            str(affiliations).split(";"),
        )
    )
    if author_id.strip() and affiliation.strip()
]

retraction_author = pd.DataFrame(records)

retraction_author.head()

,doi,author_id,is_1st_author,affiliation_country
0,10.1088/1742-6596/1852/3/032026,57223050090,True,China
1,10.1145/3465631.3465679,57215903533,True,China
2,10.1145/3465631.3465679,57216579080,False,China
3,10.1145/3465631.3465679,57214937749,False,China
4,10.3233/jifs-213414,57423073700,True,China


In [903]:
import pandas as pd

# ... [Đoạn code tạo records và retraction_subject của bạn] ...

# 1. Trích xuất dữ liệu
data = df_rwd_merged[["doi", "subject"]].dropna()

records = [
    {
        "doi": doi,
        "subject": subj.strip(),
    }
    for doi, subjects in zip(
        data["doi"],
        data["subject"],
    )
    for subj in str(subjects).split(";")
    if subj.strip()
]

retraction_subject = pd.DataFrame(records)

# 2. Xử lý xóa (XXX) và dấu cách phía sau bằng Regex
# Pattern: ^\s*\([^)]*\)\s*
# ^ : Bắt đầu chuỗi
# \s* : Bất kỳ khoảng trắng nào (nếu có)
# \([^)]*\) : Tìm cụm trong ngoặc đơn
# \s* : Xóa luôn khoảng trắng ngay sau dấu ngoặc
retraction_subject["subject"] = retraction_subject["subject"].str.replace(
    r"^\s*\([^)]*\)\s*", "", regex=True
)

# 3. (Optional) Loại bỏ các dòng bị trống sau khi đã xóa (nếu có)
retraction_subject = retraction_subject[retraction_subject["subject"] != ""]

retraction_subject.head()

,doi,subject
0,10.1088/1742-6596/1852/3/032026,Technology
1,10.1088/1742-6596/1852/3/032026,Education
2,10.1088/1742-6596/1852/3/032026,Sports and Recreation
3,10.1145/3465631.3465679,Business - Economics
4,10.1145/3465631.3465679,Business - Management


In [904]:
validation_config = {
    "retraction": {"df": retraction, "pk": ["doi"]},
    "author": {"df": author, "pk": ["author_id"]},
    "reason": {"df": reason, "pk": ["reason_tag"]},
    "retraction_author": {
        "df": retraction_author,
        "pk": ["doi", "author_id"],
    },
    "retraction_reason": {
        "df": retraction_reason,
        "pk": ["doi", "reason_tag"],
    },
    "retraction_subject": {
        "df": retraction_subject,
        "pk": ["doi", "subject"],
    },
}

for name, config in validation_config.items():
    df_target = config["df"]
    pk_columns = config["pk"]

    if df_target[pk_columns].isnull().any().any():
        bad_rows = df_target[df_target[pk_columns].isnull().any(axis=1)]
        print(f"--- Integrity Error: Null Primary Keys in '{name}' ---")
        print(bad_rows)
        raise ValueError(
            f"Null values detected in primary keys of table '{name}'."
        )

    if df_target.duplicated(subset=pk_columns).any():
        bad_rows = df_target[
            df_target.duplicated(subset=pk_columns, keep=False)
        ]
        print(f"--- Integrity Error: Duplicate Primary Keys in '{name}' ---")
        print(bad_rows)
        raise ValueError(
            f"Duplicate primary keys detected in table '{name}'."
        )

    if df_target.duplicated().any():
        bad_rows = df_target[df_target.duplicated(keep=False)]
        print(f"--- Integrity Error: Duplicate Records in '{name}' ---")
        print(bad_rows)
        raise ValueError(f"Duplicate record rows detected in table '{name}'.")

    if df_target.isnull().any().any():
        bad_rows = df_target[df_target.isnull().any(axis=1)]
        print(f"--- Integrity Error: Null Values Found in '{name}' ---")
        print(bad_rows)
        raise ValueError(
            f"Null records or missing values detected in table '{name}'."
        )

# **Data Export**

In [439]:
retraction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501 entries, 0 to 500
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   doi                  501 non-null    object        
 1   publication_date     501 non-null    datetime64[ns]
 2   retraction_date      501 non-null    datetime64[ns]
 3   journal              501 non-null    object        
 4   publisher            501 non-null    object        
 5   retraction_lag_days  501 non-null    int64         
dtypes: datetime64[ns](2), int64(1), object(3)
memory usage: 23.6+ KB


In [433]:
total_tables = len(validation_config)

for i, (name, config) in enumerate(validation_config.items(), 1):
    file_path = os.path.join(
        OUTPUT_PATH,
        f"{name}.parquet")
    
    print(f"[{i}/{total_tables}] Exporting {name}.parquet...")

    config["df"].to_parquet(
        file_path,
        engine="fastparquet",
        index=False)

print("All tables successfully exported. Done!")

[1/6] Exporting retraction.parquet...
[2/6] Exporting author.parquet...
[3/6] Exporting reason.parquet...
[4/6] Exporting retraction_author.parquet...
[5/6] Exporting retraction_reason.parquet...
[6/6] Exporting retraction_subject.parquet...
All tables successfully exported. Done!
